In [1]:
"""SAOCP evaluation on four energy datasets, with leakage-free normalization.

Implementation of Bhatnagar, Wang, Xiong & Bai (ICML 2023), Algorithm 1
(SAOCP) with Algorithm 2 (SF-OGD) and the implementation choices used by
https://github.com/salesforce/online_conformal/blob/main/online_conformal/saocp.py

Changes from the paper's forecasting experiment:
  - Base model: user's rolling LSTM lower/upper quantile regressors.
  - Nested prediction set: [q_lower - radius, q_upper + radius].
  - True nonnegative score: max(q_lower-y, y-q_upper, 0).
  - One-step-ahead sequential evaluation rather than M4 batched forecasts.
  - Scale selected from initial calibration residuals as in the authors' code.

Run from your original experiment directory, with Datasets and Helper_Scripts
in their original relative locations:
    python run_saocp_energy.py
    # Edit DEFAULT_DATASETS and configuration constants below to customize.

The actual datasets and Helper_Scripts are not embedded in the uploaded .ipynb.
Requires numpy, pandas, scikit-learn, torch, and your models_TLMR.py.
The original Helper_Scripts.calculate_metrics.py is REQUIRED for exact metric parity.
"""
from __future__ import annotations

import math
import random
import time
import sys
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd


SEED = 42
SEQ_LEN = 48
EPOCHS = 200
LR = 0.001
BATCH_SIZE = 32
ALPHA = 0.10
SPLIT = 0.50
LIFETIME = 8             # g in paper Eq. (8), authors' official default
TRAIN_EVERY = 6           # preserve notebook update cadence
DOWNSAMPLE_STRIDE = 4     # grid-only y[::4] (same as pasted four-dataset code)
SAVE_TRAJECTORIES = True
# All settings are configured here. No command-line arguments are required.
TINIT_OVERRIDE = 5000       # same fixed TINIT as the four-dataset experiment
SCALER_MODE = "prefix"     # fit only on observations available before online evaluation
DEVICE = None             # None selects CUDA automatically when available
LOG_EVERY = 200

# The local window and CWC eta are imported from the SAME helper as ACI/LACI.
# Do not define alternate local-metric formulas here.
TRAJECTORY_DIR = "./trajectories/LSTM_SAOCP_4datasets"
OUTPUT_CSV = "./results_SAOCP_4datasets.csv"
RUN_SELF_TEST = False      # True runs internal algorithm checks instead of experiments

# DEFAULT_DATASETS = [
#     "../Datasets/50Hertz.csv",
#     "../Datasets/Amprion.csv",
#     "../Datasets/TenneTTSO.csv",
#     "../Datasets/TransnetBW.csv",
# ]

DEFAULT_DATASETS = [
    "../Datasets/50Hertz.csv",
    "../Datasets/Amprion.csv",
    "../Datasets/TenneTTSO.csv",
    "../Datasets/TransnetBW.csv",
]



GRID_NAMES = {"50Hertz", "Amprion", "TenneTTSO", "TransnetBW"}


def pinball(y: float, s: float, coverage: float) -> float:
    """Paper (1): (1-alpha)-pinball loss with y=true score, s=radius."""
    return max(coverage * (y - s), (1.0 - coverage) * (s - y))


def pinball_grad(y: float, s: float, coverage: float) -> float:
    """Authors' tie convention: zero subgradient when y == s."""
    return (-coverage if y > s else (1.0 - coverage if y < s else 0.0))


def two_adic_valuation(n: int) -> int:
    assert n >= 1
    return (n & -n).bit_length() - 1


@dataclass
class SFOGDExpert:
    """Individual SAOCP expert, based on the authors' _OGD implementation."""
    start: int
    scale: float
    alpha: float
    radius: float
    g: int
    grad_norm: float = 0.0
    z: float = 0.0
    wz: float = 0.0
    age: int = 0

    @property
    def lifetime(self) -> int:
        return self.g * (2 ** two_adic_valuation(self.start))

    @property
    def expired(self) -> bool:
        # Matches author's code: expired when age > lifetime.
        return self.age > self.lifetime

    @property
    def wealth_weight(self) -> float:
        return 0.0 if self.age == 0 else (self.z / self.age) * (1.0 + self.wz)

    def update(self, score: float, meta_loss: float) -> None:
        old_weight = self.wealth_weight
        loss_expert = pinball(score, self.radius, 1.0 - self.alpha)
        # Matches official _OGD.update, including its normalization and clipping.
        gain = (meta_loss - loss_expert) / self.scale / max(self.alpha, 1.0 - self.alpha)
        gain = float(np.clip(gain, -float(old_weight > 0.0), 1.0))
        self.z += gain
        self.wz += gain * old_weight
        self.age += 1
        grad = pinball_grad(score, self.radius, 1.0 - self.alpha)
        self.grad_norm += grad * grad
        if self.grad_norm != 0.0:
            eta = self.scale / math.sqrt(3.0)
            self.radius = max(0.0, self.radius - eta * grad / math.sqrt(self.grad_norm))


class SAOCP:
    """One-step SAOCP, faithful to salesforce/online_conformal/saocp.py.

    Call predict() before seeing the current y. Then call update(score).
    update() creates an expert at the current time, aggregates meta loss,
    updates active experts and increments time; it never sees future scores.
    """
    def __init__(self, alpha: float, scale: float, lifetime: int = LIFETIME):
        if not 0.0 < alpha < 1.0:
            raise ValueError("alpha must lie strictly between zero and one")
        if not np.isfinite(scale) or scale <= 0.0:
            raise ValueError("scale must be finite and positive")
        if lifetime < 1:
            raise ValueError("lifetime must be positive")
        self.alpha = float(alpha)
        self.scale = float(scale)
        self.lifetime = int(lifetime)
        self.t = 1
        self.experts: dict[int, SFOGDExpert] = {}

    def probabilities(self) -> dict[int, float]:
        if not self.experts:
            return {}
        prior = {
            k: 1.0 / (k * k * (1.0 + math.floor(math.log2(k))))
            for k in self.experts
        }
        total_prior = sum(prior.values())
        prior = {k: v / total_prior for k, v in prior.items()}
        unnorm = {k: prior[k] * max(0.0, e.wealth_weight)
                  for k, e in self.experts.items()}
        total = sum(unnorm.values())
        return ({k: v / total for k, v in unnorm.items()}
                if total > 0.0 else prior)

    def predict(self) -> float:
        p = self.probabilities()
        return float(sum(p[k] * self.experts[k].radius for k in p))

    def update(self, score: float) -> None:
        if not np.isfinite(score) or score < 0.0:
            raise ValueError(f"Expected a nonnegative finite conformity score: {score}")
        # Official code initializes the new expert from the CURRENT aggregate,
        # then aggregates once more INCLUDING the newly created expert.
        init_radius = self.predict()
        for k in [k for k, v in self.experts.items() if v.expired]:
            del self.experts[k]
        self.experts[self.t] = SFOGDExpert(
            start=self.t, scale=self.scale, alpha=self.alpha,
            radius=init_radius, g=self.lifetime)
        meta_radius = self.predict()
        meta_loss = pinball(float(score), meta_radius, 1.0 - self.alpha)
        for expert in self.experts.values():
            expert.update(float(score), meta_loss)
        self.t += 1


def nonconformity(y: np.ndarray, lower: np.ndarray, upper: np.ndarray) -> np.ndarray:
    """Smallest nonnegative expansion of [lower, upper] covering y."""
    return np.maximum.reduce([lower - y, y - upper, np.zeros_like(y)])


def read_series(filepath: str) -> np.ndarray:
    path = Path(filepath)
    if not path.is_file():
        raise FileNotFoundError(f"Missing dataset: {path}")
    df = pd.read_csv(path)
    name = path.stem
    if name not in GRID_NAMES:
        raise ValueError(f"Only the four grid datasets are supported: {path}")
    # Match the grid preprocessing: flatten all but the first column, then downsample.
    y = df.iloc[:, 1:].to_numpy().flatten().astype(float)[::DOWNSAMPLE_STRIDE]
    if not len(y) or not np.all(np.isfinite(y)):
        raise ValueError(f"Nonfinite or empty series: {path}")
    return y


def build_sequences(series: np.ndarray, seq_len: int) -> tuple[np.ndarray, np.ndarray]:
    # A sliding view avoids the very slow Python list construction in notebook.
    if len(series) <= seq_len:
        raise ValueError("Series too short for SEQ_LEN")
    windows = np.lib.stride_tricks.sliding_window_view(series, seq_len + 1)
    X = np.ascontiguousarray(windows[:, :-1], dtype=np.float32)
    y = np.asarray(windows[:, -1], dtype=np.float32)
    return X, y


def seed_everything(seed: int) -> None:
    import torch
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def setup_models(device: str):
    import torch
    if str(Path.cwd()) not in sys.path:
        sys.path.insert(0, str(Path.cwd()))
    try:
        from Helper_Scripts.models_TLMR import LSTMQuantileReg
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            "Missing Helper_Scripts.models_TLMR.LSTMQuantileReg; run alongside "
            "your existing experiment or adjust import in setup_models()."
        ) from exc
    models = [LSTMQuantileReg().to(device) for _ in range(2)]
    opts = [torch.optim.Adam(m.parameters(), lr=LR) for m in models]
    return models, opts


def fit_models(models, opts, X: np.ndarray, y: np.ndarray, epochs: int,
               device: str, alpha: float) -> None:
    import torch
    from torch.utils.data import DataLoader, TensorDataset
    if epochs < 1:
        return
    data = TensorDataset(torch.as_tensor(X, dtype=torch.float32),
                         torch.as_tensor(y.reshape(-1, 1), dtype=torch.float32))
    loader = DataLoader(data, batch_size=min(BATCH_SIZE, len(data)), shuffle=True)
    for model, opt, q in zip(models, opts, [alpha / 2.0, 1.0 - alpha / 2.0]):
        model.train()
        for _ in range(epochs):
            for bx, by in loader:
                bx, by = bx.to(device), by.to(device)
                pred = model(bx)
                diff = by - pred
                loss = torch.maximum(q * diff, (q - 1.0) * diff).mean()
                opt.zero_grad()
                loss.backward()
                opt.step()


def predict_quantiles(models, X: np.ndarray, device: str,
                      batch_size: int = 4096) -> tuple[np.ndarray, np.ndarray]:
    import torch
    outputs = []
    for model in models:
        model.eval()
        chunks = []
        with torch.no_grad():
            for idx in range(0, len(X), batch_size):
                bx = torch.as_tensor(X[idx:idx + batch_size], dtype=torch.float32,
                                     device=device)
                chunks.append(model(bx).detach().cpu().numpy().reshape(-1))
        outputs.append(np.concatenate(chunks).astype(float))
    # Crossing correction to guarantee a valid nested interval family.
    return np.minimum(outputs[0], outputs[1]), np.maximum(outputs[0], outputs[1])


def rolling_local_trajectories(traj: pd.DataFrame, window: int, alpha: float) -> None:
    """Store per-time local diagnostics without changing the metric definitions.

    Only complete, trailing windows are reported (the first window-1 rows
    remain NaN), matching the usual local-coverage evaluation convention.
    Summary metrics are computed exclusively by the original shared helper.
    """
    if window < 1:
        raise ValueError("WINDOW from calculate_metrics must be positive")
    covered = ((traj["y_l_pred"] <= traj["y_true"]) &
               (traj["y_true"] <= traj["y_u_pred"])).astype(float)
    width = traj["y_u_pred"] - traj["y_l_pred"]
    lp = covered.rolling(window=window, min_periods=window).mean()
    traj["LPICP"] = lp
    traj["LE"] = ((1.0 - alpha) - lp).clip(lower=0.0)
    traj["local_overcoverage"] = (lp - (1.0 - alpha)).clip(lower=0.0)
    traj["local_abs_coverage_error"] = (lp - (1.0 - alpha)).abs()
    traj["local_MPIW"] = width.rolling(window=window, min_periods=window).mean()


def run_dataset(filepath: str) -> dict:
    from sklearn.preprocessing import StandardScaler
    from Helper_Scripts.calculate_metrics import (
        summarize_all_metrics, WINDOW as LOCAL_WINDOW, CWC_ETA
    )
    import torch
    name = Path(filepath).stem
    start_time = time.monotonic()
    seed_everything(SEED)
    raw = read_series(filepath)
    tinit = TINIT_OVERRIDE
    if len(raw) <= SEQ_LEN + tinit:
        raise ValueError(f"{name}: {len(raw)} points cannot support tinit={tinit}, seq_len={SEQ_LEN}")

    # Prefix mode prevents leakage from future observations into normalization.
    scaler = StandardScaler()
    scaler.fit(raw[:SEQ_LEN + tinit].reshape(-1, 1))
    series = scaler.transform(raw.reshape(-1, 1)).ravel()
    X, y = build_sequences(series, SEQ_LEN)
    n_train = int(round(SPLIT * tinit))
    if n_train < 2 or n_train >= tinit:
        raise ValueError("Train/calibration split must leave both nonempty")

    device = DEVICE or ("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n{name}: raw={len(raw)} sequences={len(y)} tinit={tinit} "
          f"training={n_train} initial_cal={tinit-n_train} device={device}", flush=True)
    models, opts = setup_models(device)
    fit_models(models, opts, X[:n_train], y[:n_train], EPOCHS, device, ALPHA)

    # Initial calibration is temporal and held out from LSTM fitting.
    cal_lo, cal_hi = predict_quantiles(models, X[n_train:tinit], device)
    cal_scores = nonconformity(y[n_train:tinit], cal_lo, cal_hi)
    max_score = float(np.max(cal_scores))
    # Authors' scale rule applied to the modified quantile-interval scores.
    # Zero-only calibration needs a strictly positive numerical fallback.
    scale = max(max_score * math.sqrt(3), 1e-8)
    saocp = SAOCP(alpha=ALPHA, scale=scale, lifetime=LIFETIME)
    for score in cal_scores:
        saocp.update(float(score))

    rows = []
    for t in range(tinit, len(y)):
        # Keep the existing periodic base-model fine-tuning protocol.
        if t > tinit and (t - tinit) % TRAIN_EVERY == 0:
            a = t - tinit
            fit_models(models, opts, X[a:a+n_train], y[a:a+n_train],
                       1, device, ALPHA)

        lo_q, hi_q = predict_quantiles(models, X[t:t+1], device)
        lo_q, hi_q = float(lo_q[0]), float(hi_q[0])
        radius = saocp.predict()  # BEFORE seeing y[t]
        lower = lo_q - radius
        upper = hi_q + radius
        yt = float(y[t])
        err = int(yt < lower or yt > upper)
        score = float(max(lo_q - yt, yt - hi_q,0.0))
        rows.append((t - tinit, t, yt, lower, upper, lo_q, hi_q,
                     radius, score, err, len(saocp.experts)))
        saocp.update(score)       # AFTER observing y[t]
        if LOG_EVERY and (t - tinit + 1) % LOG_EVERY == 0:
            print(f"  {name}: {t-tinit+1}/{len(y)-tinit} | "
                  f"radius={radius:.5f} | running_coverage="
                  f"{1.0-np.mean([r[9] for r in rows]):.4f}", flush=True)

    columns = ["step", "dataset_index", "y_true", "y_l_pred", "y_u_pred",
               "q_lower", "q_upper", "radius", "score", "err", "active_experts"]
    traj = pd.DataFrame(rows, columns=columns)
    for col in ["y_true", "y_l_pred", "y_u_pred", "q_lower", "q_upper", "radius", "score"]:
        # Transform score and radius as distances: multiply by std only.
        if col in {"radius", "score"}:
            traj[col] = traj[col] * float(scaler.scale_[0])
        else:
            traj[col] = scaler.inverse_transform(traj[[col]].to_numpy()).reshape(-1)
    traj["covered"] = 1 - traj["err"]
    traj["alphaTrajectory"] = np.nan  # SAOCP learns a radius, not alpha_t.
    traj["errSeq"] = traj["err"]
    rolling_local_trajectories(traj, LOCAL_WINDOW, ALPHA)

    outdir = Path(TRAJECTORY_DIR)
    outdir.mkdir(parents=True, exist_ok=True)
    if SAVE_TRAJECTORIES:
        traj.to_csv(outdir / f"{name}__SAOCP.csv", index=False)

    truth = traj["y_true"].to_numpy(dtype=float)
    lower = traj["y_l_pred"].to_numpy(dtype=float)
    upper = traj["y_u_pred"].to_numpy(dtype=float)
    elapsed = time.monotonic() - start_time
    # Identical evaluator, arguments and metric names as the ACI/LACI notebook.
    metrics = summarize_all_metrics(
        truth, lower, upper,
        alpha=ALPHA, w=LOCAL_WINDOW, eta=CWC_ETA, time_taken=elapsed
    )
    metrics["time_sec"] = float(elapsed)
    metrics["dataset"] = name + ".csv"
    metrics["method"] = "SAOCP"
    metrics["scale_D_from_cal"] = float(max_score * scaler.scale_[0])
    metrics["g_lifetime"] = LIFETIME
    metric_names = ["PICP", "MPIW", "MLPICP", "LPICE", "LPIA",
                    "LMPIW", "LCWC", "STD_LE"]
    print(f"[{name}/SAOCP] " + " | ".join(
        f"{k}={float(metrics[k]):.4f}" for k in metric_names if k in metrics
    ) + f" | time_sec={elapsed:.2f}", flush=True)
    return metrics


def self_test() -> None:
    # Pure NumPy/standard-library tests; no real datasets or GPU needed.
    np.random.seed(0)
    a = SAOCP(alpha=0.1, scale=math.sqrt(3), lifetime=8)
    scores = np.r_[np.full(70, 0.5), np.full(70, 2.0), np.full(70, 0.3)]
    rads = []
    for s in scores:
        r = a.predict()
        assert np.isfinite(r) and r >= 0
        rads.append(r)
        a.update(float(s))
        p = a.probabilities()
        assert abs(sum(p.values()) - 1.0) < 1e-9
        assert all(v >= 0.0 for v in p.values())
    assert len(a.experts) < 100
    assert np.mean(rads[100:140]) > np.mean(rads[25:60]), "Radius failed to respond to shift"
    assert np.allclose(nonconformity(np.array([2., 5., 7.]),
                                      np.array([1., 4., 4.]),
                                      np.array([3., 6., 6.])), [0., 0., 1.])
    # The nested set coverage event is score <= radius, including equality.
    for s in (0.0, 0.5, 2.0):
        for rad in (0.0, 0.5, 2.0):
            assert ((10.0 - rad <= 10.0 + s <= 10.0 + rad) == (s <= rad))
    print("Self-test passed: expert probabilities, finite radii, shift response, scores, interval inclusion")


def main() -> None:
    if RUN_SELF_TEST:
        self_test()
        return
    if TRAIN_EVERY < 1:
        raise ValueError("TRAIN_EVERY must be >= 1")
    if SCALER_MODE != "prefix":
        raise ValueError("Only leakage-free SCALER_MODE='prefix' is supported")
    if DOWNSAMPLE_STRIDE < 1:
        raise ValueError("DOWNSAMPLE_STRIDE must be >= 1")
    if not DEFAULT_DATASETS:
        raise ValueError("DEFAULT_DATASETS must include at least one file")
    missing = [p for p in DEFAULT_DATASETS if not Path(p).is_file()]
    if missing:
        raise FileNotFoundError("Datasets missing (no synthetic fallback):\n" + "\n".join(missing))
    metrics = []
    for path in DEFAULT_DATASETS:
        row = run_dataset(path)
        metrics.append(row)
        pd.DataFrame(metrics).to_csv(OUTPUT_CSV, index=False)
    # Match the column ordering in your ACI/DtACI/AgACI/LACI notebook.
    results_df = pd.DataFrame(metrics)
    metric_order = ["dataset", "method", "PICP", "MPIW", "MLPICP", "LPICE",
                    "LPIA", "LMPIW", "LCWC", "STD_LE", "n_local_windows",
                    "time_sec"]
    ordered = [k for k in metric_order if k in results_df.columns]
    results_df = results_df[ordered + [k for k in results_df if k not in ordered]]
    results_df.to_csv(OUTPUT_CSV, index=False)
    print("\nSaved summary:", OUTPUT_CSV)
    if SAVE_TRAJECTORIES:
        print("Saved trajectories:", TRAJECTORY_DIR)
    print(results_df.to_string(index=False))


if __name__ == "__main__":
    main()



50Hertz: raw=9528 sequences=9480 tinit=5000 training=2500 initial_cal=2500 device=cuda
  50Hertz: 200/4480 | radius=0.00791 | running_coverage=0.9050
  50Hertz: 400/4480 | radius=0.03973 | running_coverage=0.9050
  50Hertz: 600/4480 | radius=0.04642 | running_coverage=0.9067
  50Hertz: 800/4480 | radius=0.11963 | running_coverage=0.9087
  50Hertz: 1000/4480 | radius=0.19515 | running_coverage=0.9110
  50Hertz: 1200/4480 | radius=0.00000 | running_coverage=0.9158
  50Hertz: 1400/4480 | radius=0.01938 | running_coverage=0.9164
  50Hertz: 1600/4480 | radius=0.08637 | running_coverage=0.9138
  50Hertz: 1800/4480 | radius=0.02902 | running_coverage=0.9150
  50Hertz: 2000/4480 | radius=0.00000 | running_coverage=0.9160
  50Hertz: 2200/4480 | radius=0.05600 | running_coverage=0.9173
  50Hertz: 2400/4480 | radius=0.09342 | running_coverage=0.9158
  50Hertz: 2600/4480 | radius=0.19466 | running_coverage=0.9154
  50Hertz: 2800/4480 | radius=0.05168 | running_coverage=0.9161
  50Hertz: 3000/4480

In [2]:
"""SAOCP evaluation on Location1 and Location2, with leakage-free normalization.

Implementation of Bhatnagar, Wang, Xiong & Bai (ICML 2023), Algorithm 1
(SAOCP) with Algorithm 2 (SF-OGD) and the implementation choices used by
https://github.com/salesforce/online_conformal/blob/main/online_conformal/saocp.py

Changes from the paper's forecasting experiment:
  - Base model: user's rolling LSTM lower/upper quantile regressors.
  - Nested prediction set: [q_lower - radius, q_upper + radius].
  - True nonnegative score: max(q_lower-y, y-q_upper, 0).
  - One-step-ahead sequential evaluation rather than M4 batched forecasts.
  - Scale selected from initial calibration residuals as in the authors' code.

Run from your original experiment directory, with Datasets and Helper_Scripts
in their original relative locations:
    python run_saocp_two_locations_corrected.py
    # Edit DEFAULT_DATASETS and configuration constants below to customize.

The actual datasets and Helper_Scripts are not embedded in the uploaded .ipynb.
Requires numpy, pandas, scikit-learn, torch, and your models_TLMR.py.
The original Helper_Scripts.calculate_metrics.py is REQUIRED for exact metric parity.
"""
from __future__ import annotations

import math
import random
import time
import sys
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd


SEED = 42
SEQ_LEN = 48
EPOCHS = 200
LR = 0.001
BATCH_SIZE = 32
ALPHA = 0.10
SPLIT = 0.50
LIFETIME = 8             # g in paper Eq. (8), authors' official default
TRAIN_EVERY = 6           # preserve notebook update cadence
SAVE_TRAJECTORIES = True
# All settings are configured here. No command-line arguments are required.
TINIT_OVERRIDE = 5000       # same fixed TINIT as the two-location experiment
SCALER_MODE = "prefix"     # fit only on observations available before online evaluation
DEVICE = None             # None selects CUDA automatically when available
LOG_EVERY = 200

# The local window and CWC eta are imported from the SAME helper as ACI/LACI.
# Do not define alternate local-metric formulas here.
TRAJECTORY_DIR = "./trajectories/LSTM_SAOCP_2locations"
OUTPUT_CSV = "./results_SAOCP_2locations.csv"
RUN_SELF_TEST = False      # True runs internal algorithm checks instead of experiments

DEFAULT_DATASETS = [
    "../Datasets/Location1.csv",
    "../Datasets/Location2.csv",
]

LOCATION_NAMES = {"Location1", "Location2"}


def pinball(y: float, s: float, coverage: float) -> float:
    """Paper (1): (1-alpha)-pinball loss with y=true score, s=radius."""
    return max(coverage * (y - s), (1.0 - coverage) * (s - y))


def pinball_grad(y: float, s: float, coverage: float) -> float:
    """Authors' tie convention: zero subgradient when y == s."""
    return (-coverage if y > s else (1.0 - coverage if y < s else 0.0))


def two_adic_valuation(n: int) -> int:
    assert n >= 1
    return (n & -n).bit_length() - 1


@dataclass
class SFOGDExpert:
    """Individual SAOCP expert, based on the authors' _OGD implementation."""
    start: int
    scale: float
    alpha: float
    radius: float
    g: int
    grad_norm: float = 0.0
    z: float = 0.0
    wz: float = 0.0
    age: int = 0

    @property
    def lifetime(self) -> int:
        return self.g * (2 ** two_adic_valuation(self.start))

    @property
    def expired(self) -> bool:
        # Matches author's code: expired when age > lifetime.
        return self.age > self.lifetime

    @property
    def wealth_weight(self) -> float:
        return 0.0 if self.age == 0 else (self.z / self.age) * (1.0 + self.wz)

    def update(self, score: float, meta_loss: float) -> None:
        old_weight = self.wealth_weight
        loss_expert = pinball(score, self.radius, 1.0 - self.alpha)
        # Matches official _OGD.update, including its normalization and clipping.
        gain = (meta_loss - loss_expert) / self.scale / max(self.alpha, 1.0 - self.alpha)
        gain = float(np.clip(gain, -float(old_weight > 0.0), 1.0))
        self.z += gain
        self.wz += gain * old_weight
        self.age += 1
        grad = pinball_grad(score, self.radius, 1.0 - self.alpha)
        self.grad_norm += grad * grad
        if self.grad_norm != 0.0:
            eta = self.scale / math.sqrt(3.0)
            self.radius = max(0.0, self.radius - eta * grad / math.sqrt(self.grad_norm))


class SAOCP:
    """One-step SAOCP, faithful to salesforce/online_conformal/saocp.py.

    Call predict() before seeing the current y. Then call update(score).
    update() creates an expert at the current time, aggregates meta loss,
    updates active experts and increments time; it never sees future scores.
    """
    def __init__(self, alpha: float, scale: float, lifetime: int = LIFETIME):
        if not 0.0 < alpha < 1.0:
            raise ValueError("alpha must lie strictly between zero and one")
        if not np.isfinite(scale) or scale <= 0.0:
            raise ValueError("scale must be finite and positive")
        if lifetime < 1:
            raise ValueError("lifetime must be positive")
        self.alpha = float(alpha)
        self.scale = float(scale)
        self.lifetime = int(lifetime)
        self.t = 1
        self.experts: dict[int, SFOGDExpert] = {}

    def probabilities(self) -> dict[int, float]:
        if not self.experts:
            return {}
        prior = {
            k: 1.0 / (k * k * (1.0 + math.floor(math.log2(k))))
            for k in self.experts
        }
        total_prior = sum(prior.values())
        prior = {k: v / total_prior for k, v in prior.items()}
        unnorm = {k: prior[k] * max(0.0, e.wealth_weight)
                  for k, e in self.experts.items()}
        total = sum(unnorm.values())
        return ({k: v / total for k, v in unnorm.items()}
                if total > 0.0 else prior)

    def predict(self) -> float:
        p = self.probabilities()
        return float(sum(p[k] * self.experts[k].radius for k in p))

    def update(self, score: float) -> None:
        if not np.isfinite(score) or score < 0.0:
            raise ValueError(f"Expected a nonnegative finite conformity score: {score}")
        # Official code initializes the new expert from the CURRENT aggregate,
        # then aggregates once more INCLUDING the newly created expert.
        init_radius = self.predict()
        for k in [k for k, v in self.experts.items() if v.expired]:
            del self.experts[k]
        self.experts[self.t] = SFOGDExpert(
            start=self.t, scale=self.scale, alpha=self.alpha,
            radius=init_radius, g=self.lifetime)
        meta_radius = self.predict()
        meta_loss = pinball(float(score), meta_radius, 1.0 - self.alpha)
        for expert in self.experts.values():
            expert.update(float(score), meta_loss)
        self.t += 1


def nonconformity(y: np.ndarray, lower: np.ndarray, upper: np.ndarray) -> np.ndarray:
    """Smallest nonnegative expansion of [lower, upper] covering y."""
    return np.maximum.reduce([lower - y, y - upper, np.zeros_like(y)])


def read_series(filepath: str) -> np.ndarray:
    """Match the ACI two-location loader: first 10000 rows, Power column."""
    path = Path(filepath)
    if not path.is_file():
        raise FileNotFoundError(f"Missing dataset: {path}")
    if path.stem not in LOCATION_NAMES:
        raise ValueError(f"Only Location1 and Location2 are supported: {path}")
    df = pd.read_csv(path).iloc[:10000]
    if "Power" not in df.columns:
        raise ValueError(f"{path}: required 'Power' column is missing")
    y = pd.to_numeric(df["Power"], errors="raise").to_numpy(dtype=float)
    if not len(y) or not np.all(np.isfinite(y)):
        raise ValueError(f"Nonfinite or empty Power series: {path}")
    return y


def build_sequences(series: np.ndarray, seq_len: int) -> tuple[np.ndarray, np.ndarray]:
    # A sliding view avoids the very slow Python list construction in notebook.
    if len(series) <= seq_len:
        raise ValueError("Series too short for SEQ_LEN")
    windows = np.lib.stride_tricks.sliding_window_view(series, seq_len + 1)
    X = np.ascontiguousarray(windows[:, :-1], dtype=np.float32)
    y = np.asarray(windows[:, -1], dtype=np.float32)
    return X, y


def seed_everything(seed: int) -> None:
    import torch
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def setup_models(device: str):
    import torch
    if str(Path.cwd()) not in sys.path:
        sys.path.insert(0, str(Path.cwd()))
    try:
        from Helper_Scripts.models_TLMR import LSTMQuantileReg
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            "Missing Helper_Scripts.models_TLMR.LSTMQuantileReg; run alongside "
            "your existing experiment or adjust import in setup_models()."
        ) from exc
    models = [LSTMQuantileReg().to(device) for _ in range(2)]
    opts = [torch.optim.Adam(m.parameters(), lr=LR) for m in models]
    return models, opts


def fit_models(models, opts, X: np.ndarray, y: np.ndarray, epochs: int,
               device: str, alpha: float) -> None:
    import torch
    from torch.utils.data import DataLoader, TensorDataset
    if epochs < 1:
        return
    data = TensorDataset(torch.as_tensor(X, dtype=torch.float32),
                         torch.as_tensor(y.reshape(-1, 1), dtype=torch.float32))
    loader = DataLoader(data, batch_size=min(BATCH_SIZE, len(data)), shuffle=True)
    for model, opt, q in zip(models, opts, [alpha / 2.0, 1.0 - alpha / 2.0]):
        model.train()
        for _ in range(epochs):
            for bx, by in loader:
                bx, by = bx.to(device), by.to(device)
                pred = model(bx)
                diff = by - pred
                loss = torch.maximum(q * diff, (q - 1.0) * diff).mean()
                opt.zero_grad()
                loss.backward()
                opt.step()


def predict_quantiles(models, X: np.ndarray, device: str,
                      batch_size: int = 4096) -> tuple[np.ndarray, np.ndarray]:
    import torch
    outputs = []
    for model in models:
        model.eval()
        chunks = []
        with torch.no_grad():
            for idx in range(0, len(X), batch_size):
                bx = torch.as_tensor(X[idx:idx + batch_size], dtype=torch.float32,
                                     device=device)
                chunks.append(model(bx).detach().cpu().numpy().reshape(-1))
        outputs.append(np.concatenate(chunks).astype(float))
    # Crossing correction to guarantee a valid nested interval family.
    return np.minimum(outputs[0], outputs[1]), np.maximum(outputs[0], outputs[1])


def rolling_local_trajectories(traj: pd.DataFrame, window: int, alpha: float) -> None:
    """Store per-time local diagnostics without changing the metric definitions.

    Only complete, trailing windows are reported (the first window-1 rows
    remain NaN), matching the usual local-coverage evaluation convention.
    Summary metrics are computed exclusively by the original shared helper.
    """
    if window < 1:
        raise ValueError("WINDOW from calculate_metrics must be positive")
    covered = ((traj["y_l_pred"] <= traj["y_true"]) &
               (traj["y_true"] <= traj["y_u_pred"])).astype(float)
    width = traj["y_u_pred"] - traj["y_l_pred"]
    lp = covered.rolling(window=window, min_periods=window).mean()
    traj["LPICP"] = lp
    traj["LE"] = ((1.0 - alpha) - lp).clip(lower=0.0)
    traj["local_overcoverage"] = (lp - (1.0 - alpha)).clip(lower=0.0)
    traj["local_abs_coverage_error"] = (lp - (1.0 - alpha)).abs()
    traj["local_MPIW"] = width.rolling(window=window, min_periods=window).mean()


def run_dataset(filepath: str) -> dict:
    from sklearn.preprocessing import StandardScaler
    from Helper_Scripts.calculate_metrics import (
        summarize_all_metrics, WINDOW as LOCAL_WINDOW, CWC_ETA
    )
    import torch
    name = Path(filepath).stem
    start_time = time.monotonic()
    seed_everything(SEED)
    raw = read_series(filepath)
    tinit = TINIT_OVERRIDE
    if len(raw) <= SEQ_LEN + tinit:
        raise ValueError(f"{name}: {len(raw)} points cannot support tinit={tinit}, seq_len={SEQ_LEN}")

    # Prefix mode prevents leakage from future observations into normalization.
    scaler = StandardScaler()
    scaler.fit(raw[:SEQ_LEN + tinit].reshape(-1, 1))
    series = scaler.transform(raw.reshape(-1, 1)).ravel()
    X, y = build_sequences(series, SEQ_LEN)
    n_train = int(round(SPLIT * tinit))
    if n_train < 2 or n_train >= tinit:
        raise ValueError("Train/calibration split must leave both nonempty")

    device = DEVICE or ("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n{name}: raw={len(raw)} sequences={len(y)} tinit={tinit} "
          f"training={n_train} initial_cal={tinit-n_train} device={device}", flush=True)
    models, opts = setup_models(device)
    fit_models(models, opts, X[:n_train], y[:n_train], EPOCHS, device, ALPHA)

    # Initial calibration is temporal and held out from LSTM fitting.
    cal_lo, cal_hi = predict_quantiles(models, X[n_train:tinit], device)
    cal_scores = nonconformity(y[n_train:tinit], cal_lo, cal_hi)
    max_score = float(np.max(cal_scores))
    # Authors' scale rule applied to the modified quantile-interval scores.
    # Zero-only calibration needs a strictly positive numerical fallback.
    scale = max(max_score * math.sqrt(3), 1e-8)
    saocp = SAOCP(alpha=ALPHA, scale=scale, lifetime=LIFETIME)
    for score in cal_scores:
        saocp.update(float(score))

    rows = []
    for t in range(tinit, len(y)):
        # Keep the existing periodic base-model fine-tuning protocol.
        if t > tinit and (t - tinit) % TRAIN_EVERY == 0:
            a = t - tinit
            fit_models(models, opts, X[a:a+n_train], y[a:a+n_train],
                       1, device, ALPHA)

        lo_q, hi_q = predict_quantiles(models, X[t:t+1], device)
        lo_q, hi_q = float(lo_q[0]), float(hi_q[0])
        radius = saocp.predict()  # BEFORE seeing y[t]
        lower = lo_q - radius
        upper = hi_q + radius
        yt = float(y[t])
        err = int(yt < lower or yt > upper)
        score = float(max(lo_q - yt, yt - hi_q,0.0))
        rows.append((t - tinit, t, yt, lower, upper, lo_q, hi_q,
                     radius, score, err, len(saocp.experts)))
        saocp.update(score)       # AFTER observing y[t]
        if LOG_EVERY and (t - tinit + 1) % LOG_EVERY == 0:
            print(f"  {name}: {t-tinit+1}/{len(y)-tinit} | "
                  f"radius={radius:.5f} | running_coverage="
                  f"{1.0-np.mean([r[9] for r in rows]):.4f}", flush=True)

    columns = ["step", "dataset_index", "y_true", "y_l_pred", "y_u_pred",
               "q_lower", "q_upper", "radius", "score", "err", "active_experts"]
    traj = pd.DataFrame(rows, columns=columns)
    for col in ["y_true", "y_l_pred", "y_u_pred", "q_lower", "q_upper", "radius", "score"]:
        # Transform score and radius as distances: multiply by std only.
        if col in {"radius", "score"}:
            traj[col] = traj[col] * float(scaler.scale_[0])
        else:
            traj[col] = scaler.inverse_transform(traj[[col]].to_numpy()).reshape(-1)
    traj["covered"] = 1 - traj["err"]
    traj["alphaTrajectory"] = np.nan  # SAOCP learns a radius, not alpha_t.
    traj["errSeq"] = traj["err"]
    rolling_local_trajectories(traj, LOCAL_WINDOW, ALPHA)

    outdir = Path(TRAJECTORY_DIR)
    outdir.mkdir(parents=True, exist_ok=True)
    if SAVE_TRAJECTORIES:
        traj.to_csv(outdir / f"{name}__SAOCP.csv", index=False)

    truth = traj["y_true"].to_numpy(dtype=float)
    lower = traj["y_l_pred"].to_numpy(dtype=float)
    upper = traj["y_u_pred"].to_numpy(dtype=float)
    elapsed = time.monotonic() - start_time
    # Identical evaluator, arguments and metric names as the ACI/LACI notebook.
    metrics = summarize_all_metrics(
        truth, lower, upper,
        alpha=ALPHA, w=LOCAL_WINDOW, eta=CWC_ETA, time_taken=elapsed
    )
    metrics["time_sec"] = float(elapsed)
    metrics["dataset"] = name + ".csv"
    metrics["method"] = "SAOCP"
    metrics["scale_D_from_cal"] = float(max_score * scaler.scale_[0])
    metrics["g_lifetime"] = LIFETIME
    metric_names = ["PICP", "MPIW", "MLPICP", "LPICE", "LPIA",
                    "LMPIW", "LCWC", "STD_LE"]
    print(f"[{name}/SAOCP] " + " | ".join(
        f"{k}={float(metrics[k]):.4f}" for k in metric_names if k in metrics
    ) + f" | time_sec={elapsed:.2f}", flush=True)
    return metrics


def self_test() -> None:
    # Pure NumPy/standard-library tests; no real datasets or GPU needed.
    np.random.seed(0)
    a = SAOCP(alpha=0.1, scale=math.sqrt(3), lifetime=8)
    scores = np.r_[np.full(70, 0.5), np.full(70, 2.0), np.full(70, 0.3)]
    rads = []
    for s in scores:
        r = a.predict()
        assert np.isfinite(r) and r >= 0
        rads.append(r)
        a.update(float(s))
        p = a.probabilities()
        assert abs(sum(p.values()) - 1.0) < 1e-9
        assert all(v >= 0.0 for v in p.values())
    assert len(a.experts) < 100
    assert np.mean(rads[100:140]) > np.mean(rads[25:60]), "Radius failed to respond to shift"
    assert np.allclose(nonconformity(np.array([2., 5., 7.]),
                                      np.array([1., 4., 4.]),
                                      np.array([3., 6., 6.])), [0., 0., 1.])
    # The nested set coverage event is score <= radius, including equality.
    for s in (0.0, 0.5, 2.0):
        for rad in (0.0, 0.5, 2.0):
            assert ((10.0 - rad <= 10.0 + s <= 10.0 + rad) == (s <= rad))
    print("Self-test passed: expert probabilities, finite radii, shift response, scores, interval inclusion")


def main() -> None:
    if RUN_SELF_TEST:
        self_test()
        return
    if TRAIN_EVERY < 1:
        raise ValueError("TRAIN_EVERY must be >= 1")
    if SCALER_MODE != "prefix":
        raise ValueError("Only leakage-free SCALER_MODE='prefix' is supported")
    if not DEFAULT_DATASETS:
        raise ValueError("DEFAULT_DATASETS must include at least one file")
    missing = [p for p in DEFAULT_DATASETS if not Path(p).is_file()]
    if missing:
        raise FileNotFoundError("Datasets missing (no synthetic fallback):\n" + "\n".join(missing))
    metrics = []
    for path in DEFAULT_DATASETS:
        row = run_dataset(path)
        metrics.append(row)
        pd.DataFrame(metrics).to_csv(OUTPUT_CSV, index=False)
    # Match the column ordering in your ACI/DtACI/AgACI/LACI notebook.
    results_df = pd.DataFrame(metrics)
    metric_order = ["dataset", "method", "PICP", "MPIW", "MLPICP", "LPICE",
                    "LPIA", "LMPIW", "LCWC", "STD_LE", "n_local_windows",
                    "time_sec"]
    ordered = [k for k in metric_order if k in results_df.columns]
    results_df = results_df[ordered + [k for k in results_df if k not in ordered]]
    results_df.to_csv(OUTPUT_CSV, index=False)
    print("\nSaved summary:", OUTPUT_CSV)
    if SAVE_TRAJECTORIES:
        print("Saved trajectories:", TRAJECTORY_DIR)
    print(results_df.to_string(index=False))


if __name__ == "__main__":
    main()



Location1: raw=10000 sequences=9952 tinit=5000 training=2500 initial_cal=2500 device=cuda
  Location1: 200/4952 | radius=0.00000 | running_coverage=0.9800
  Location1: 400/4952 | radius=0.00230 | running_coverage=0.9700
  Location1: 600/4952 | radius=0.00000 | running_coverage=0.9717
  Location1: 800/4952 | radius=0.03541 | running_coverage=0.9725
  Location1: 1000/4952 | radius=0.00000 | running_coverage=0.9690
  Location1: 1200/4952 | radius=0.00146 | running_coverage=0.9683
  Location1: 1400/4952 | radius=0.00000 | running_coverage=0.9636
  Location1: 1600/4952 | radius=0.00288 | running_coverage=0.9625
  Location1: 1800/4952 | radius=0.00000 | running_coverage=0.9611
  Location1: 2000/4952 | radius=0.00208 | running_coverage=0.9590
  Location1: 2200/4952 | radius=0.00121 | running_coverage=0.9568
  Location1: 2400/4952 | radius=0.00000 | running_coverage=0.9567
  Location1: 2600/4952 | radius=0.05849 | running_coverage=0.9550
  Location1: 2800/4952 | radius=0.00885 | running_cover

In [3]:
"""SAOCP-only evaluation on Plant 1 and Plant 2 solar datasets, matching the supplied ACI setup.

Implementation of Bhatnagar, Wang, Xiong & Bai (ICML 2023), Algorithm 1
(SAOCP) with Algorithm 2 (SF-OGD) and the implementation choices used by
https://github.com/salesforce/online_conformal/blob/main/online_conformal/saocp.py

Changes from the paper's forecasting experiment:
  - Base model: user's rolling LSTM lower/upper quantile regressors.
  - Nested prediction set: [q_lower - radius, q_upper + radius].
  - True nonnegative score: max(q_lower-y, y-q_upper, 0).
  - One-step-ahead sequential evaluation rather than M4 batched forecasts.
  - Scale selected from initial calibration residuals as in the authors' code.

Run from your original experiment directory, with Datasets and Helper_Scripts
in their original relative locations:
    python run_saocp_two_solar_corrected.py
    # Edit DEFAULT_DATASETS and configuration constants below to customize.

The actual datasets and Helper_Scripts are not embedded in the uploaded .ipynb.
Requires numpy, pandas, scikit-learn, torch, and your models_TLMR.py.
The original Helper_Scripts.calculate_metrics.py is REQUIRED for exact metric parity.
"""
from __future__ import annotations

import math
import random
import time
import sys
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd


SEED = 42
SEQ_LEN = 48
EPOCHS = 200
LR = 0.001
BATCH_SIZE = 32
ALPHA = 0.10
SPLIT = 0.50
LIFETIME = 8              # g in paper Eq. (8), authors' official default
TRAIN_EVERY = 6           # preserve notebook update cadence
SAVE_TRAJECTORIES = True
# All settings are configured here. No command-line arguments are required.
TINIT_OVERRIDE = 4000       # same fixed TINIT as the solar experiment
SCALER_MODE = "prefix"     # fit only on observations available before online evaluation
DEVICE = None             # None selects CUDA automatically when available
LOG_EVERY = 200

# The local window and CWC eta are imported from the SAME helper as ACI/LACI.
# Do not define alternate local-metric formulas here.
TRAJECTORY_DIR = "./trajectories/LSTM_SAOCP_2solar"
OUTPUT_CSV = "./results_SAOCP_2solar.csv"
RUN_SELF_TEST = False      # True runs internal algorithm checks instead of experiments

DEFAULT_DATASETS = [
    "../Datasets/Plant_1_Generation_Data.csv",
    "../Datasets/Plant_2_Generation_Data.csv",
]

SOLAR_NAMES = {"Plant_1_Generation_Data", "Plant_2_Generation_Data"}


def pinball(y: float, s: float, coverage: float) -> float:
    """Paper (1): (1-alpha)-pinball loss with y=true score, s=radius."""
    return max(coverage * (y - s), (1.0 - coverage) * (s - y))


def pinball_grad(y: float, s: float, coverage: float) -> float:
    """Authors' tie convention: zero subgradient when y == s."""
    return (-coverage if y > s else (1.0 - coverage if y < s else 0.0))


def two_adic_valuation(n: int) -> int:
    assert n >= 1
    return (n & -n).bit_length() - 1


@dataclass
class SFOGDExpert:
    """Individual SAOCP expert, based on the authors' _OGD implementation."""
    start: int
    scale: float
    alpha: float
    radius: float
    g: int
    grad_norm: float = 0.0
    z: float = 0.0
    wz: float = 0.0
    age: int = 0

    @property
    def lifetime(self) -> int:
        return self.g * (2 ** two_adic_valuation(self.start))

    @property
    def expired(self) -> bool:
        # Matches author's code: expired when age > lifetime.
        return self.age > self.lifetime

    @property
    def wealth_weight(self) -> float:
        return 0.0 if self.age == 0 else (self.z / self.age) * (1.0 + self.wz)

    def update(self, score: float, meta_loss: float) -> None:
        old_weight = self.wealth_weight
        loss_expert = pinball(score, self.radius, 1.0 - self.alpha)
        # Matches official _OGD.update, including its normalization and clipping.
        gain = (meta_loss - loss_expert) / self.scale / max(self.alpha, 1.0 - self.alpha)
        gain = float(np.clip(gain, -float(old_weight > 0.0), 1.0))
        self.z += gain
        self.wz += gain * old_weight
        self.age += 1
        grad = pinball_grad(score, self.radius, 1.0 - self.alpha)
        self.grad_norm += grad * grad
        if self.grad_norm != 0.0:
            eta = self.scale / math.sqrt(3.0)
            self.radius = max(0.0, self.radius - eta * grad / math.sqrt(self.grad_norm))


class SAOCP:
    """One-step SAOCP, faithful to salesforce/online_conformal/saocp.py.

    Call predict() before seeing the current y. Then call update(score).
    update() creates an expert at the current time, aggregates meta loss,
    updates active experts and increments time; it never sees future scores.
    """
    def __init__(self, alpha: float, scale: float, lifetime: int = LIFETIME):
        if not 0.0 < alpha < 1.0:
            raise ValueError("alpha must lie strictly between zero and one")
        if not np.isfinite(scale) or scale <= 0.0:
            raise ValueError("scale must be finite and positive")
        if lifetime < 1:
            raise ValueError("lifetime must be positive")
        self.alpha = float(alpha)
        self.scale = float(scale)
        self.lifetime = int(lifetime)
        self.t = 1
        self.experts: dict[int, SFOGDExpert] = {}

    def probabilities(self) -> dict[int, float]:
        if not self.experts:
            return {}
        prior = {
            k: 1.0 / (k * k * (1.0 + math.floor(math.log2(k))))
            for k in self.experts
        }
        total_prior = sum(prior.values())
        prior = {k: v / total_prior for k, v in prior.items()}
        unnorm = {k: prior[k] * max(0.0, e.wealth_weight)
                  for k, e in self.experts.items()}
        total = sum(unnorm.values())
        return ({k: v / total for k, v in unnorm.items()}
                if total > 0.0 else prior)

    def predict(self) -> float:
        p = self.probabilities()
        return float(sum(p[k] * self.experts[k].radius for k in p))

    def update(self, score: float) -> None:
        if not np.isfinite(score) or score < 0.0:
            raise ValueError(f"Expected a nonnegative finite conformity score: {score}")
        # Official code initializes the new expert from the CURRENT aggregate,
        # then aggregates once more INCLUDING the newly created expert.
        init_radius = self.predict()
        for k in [k for k, v in self.experts.items() if v.expired]:
            del self.experts[k]
        self.experts[self.t] = SFOGDExpert(
            start=self.t, scale=self.scale, alpha=self.alpha,
            radius=init_radius, g=self.lifetime)
        meta_radius = self.predict()
        meta_loss = pinball(float(score), meta_radius, 1.0 - self.alpha)
        for expert in self.experts.values():
            expert.update(float(score), meta_loss)
        self.t += 1


def nonconformity(y: np.ndarray, lower: np.ndarray, upper: np.ndarray) -> np.ndarray:
    """Smallest nonnegative expansion of [lower, upper] covering y."""
    return np.maximum.reduce([lower - y, y - upper, np.zeros_like(y)])


def read_series(filepath: str) -> np.ndarray:
    """Match the solar ACI loader: first 10000 rows, non-null/nonzero DAILY_YIELD."""
    path = Path(filepath)
    if not path.is_file():
        raise FileNotFoundError(f"Missing dataset: {path}")
    if path.stem not in SOLAR_NAMES:
        raise ValueError(f"Only the two solar plant datasets are supported: {path}")
    df = pd.read_csv(path).iloc[:10000]
    if "DAILY_YIELD" not in df.columns:
        raise ValueError(f"{path}: required 'DAILY_YIELD' column is missing")
    values = pd.to_numeric(df["DAILY_YIELD"], errors="coerce")
    y = values.loc[values.notna() & (values != 0)].to_numpy(dtype=float)
    # No downsampling: the original solar pipeline used the filtered series.
    if not len(y) or not np.all(np.isfinite(y)):
        raise ValueError(f"Nonfinite or empty filtered DAILY_YIELD series: {path}")
    return y


def build_sequences(series: np.ndarray, seq_len: int) -> tuple[np.ndarray, np.ndarray]:
    # A sliding view avoids the very slow Python list construction in notebook.
    if len(series) <= seq_len:
        raise ValueError("Series too short for SEQ_LEN")
    windows = np.lib.stride_tricks.sliding_window_view(series, seq_len + 1)
    X = np.ascontiguousarray(windows[:, :-1], dtype=np.float32)
    y = np.asarray(windows[:, -1], dtype=np.float32)
    return X, y


def seed_everything(seed: int) -> None:
    import torch
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def setup_models(device: str):
    import torch
    if str(Path.cwd()) not in sys.path:
        sys.path.insert(0, str(Path.cwd()))
    try:
        from Helper_Scripts.models_TLMR import LSTMQuantileReg
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            "Missing Helper_Scripts.models_TLMR.LSTMQuantileReg; run alongside "
            "your existing experiment or adjust import in setup_models()."
        ) from exc
    models = [LSTMQuantileReg().to(device) for _ in range(2)]
    opts = [torch.optim.Adam(m.parameters(), lr=LR) for m in models]
    return models, opts


def fit_models(models, opts, X: np.ndarray, y: np.ndarray, epochs: int,
               device: str, alpha: float) -> None:
    import torch
    from torch.utils.data import DataLoader, TensorDataset
    if epochs < 1:
        return
    data = TensorDataset(torch.as_tensor(X, dtype=torch.float32),
                         torch.as_tensor(y.reshape(-1, 1), dtype=torch.float32))
    loader = DataLoader(data, batch_size=min(BATCH_SIZE, len(data)), shuffle=True)
    for model, opt, q in zip(models, opts, [alpha / 2.0, 1.0 - alpha / 2.0]):
        model.train()
        for _ in range(epochs):
            for bx, by in loader:
                bx, by = bx.to(device), by.to(device)
                pred = model(bx)
                diff = by - pred
                loss = torch.maximum(q * diff, (q - 1.0) * diff).mean()
                opt.zero_grad()
                loss.backward()
                opt.step()


def predict_quantiles(models, X: np.ndarray, device: str,
                      batch_size: int = 4096) -> tuple[np.ndarray, np.ndarray]:
    import torch
    outputs = []
    for model in models:
        model.eval()
        chunks = []
        with torch.no_grad():
            for idx in range(0, len(X), batch_size):
                bx = torch.as_tensor(X[idx:idx + batch_size], dtype=torch.float32,
                                     device=device)
                chunks.append(model(bx).detach().cpu().numpy().reshape(-1))
        outputs.append(np.concatenate(chunks).astype(float))
    # Crossing correction to guarantee a valid nested interval family.
    return np.minimum(outputs[0], outputs[1]), np.maximum(outputs[0], outputs[1])


def rolling_local_trajectories(traj: pd.DataFrame, window: int, alpha: float) -> None:
    """Store per-time local diagnostics without changing the metric definitions.

    Only complete, trailing windows are reported (the first window-1 rows
    remain NaN), matching the usual local-coverage evaluation convention.
    Summary metrics are computed exclusively by the original shared helper.
    """
    if window < 1:
        raise ValueError("WINDOW from calculate_metrics must be positive")
    covered = ((traj["y_l_pred"] <= traj["y_true"]) &
               (traj["y_true"] <= traj["y_u_pred"])).astype(float)
    width = traj["y_u_pred"] - traj["y_l_pred"]
    lp = covered.rolling(window=window, min_periods=window).mean()
    traj["LPICP"] = lp
    traj["LE"] = ((1.0 - alpha) - lp).clip(lower=0.0)
    traj["local_overcoverage"] = (lp - (1.0 - alpha)).clip(lower=0.0)
    traj["local_abs_coverage_error"] = (lp - (1.0 - alpha)).abs()
    traj["local_MPIW"] = width.rolling(window=window, min_periods=window).mean()


def run_dataset(filepath: str) -> dict:
    from sklearn.preprocessing import StandardScaler
    from Helper_Scripts.calculate_metrics import (
        summarize_all_metrics, WINDOW as LOCAL_WINDOW, CWC_ETA
    )
    import torch
    name = Path(filepath).stem
    start_time = time.monotonic()
    seed_everything(SEED)
    raw = read_series(filepath)
    tinit = TINIT_OVERRIDE
    if len(raw) <= SEQ_LEN + tinit:
        raise ValueError(f"{name}: {len(raw)} points cannot support tinit={tinit}, seq_len={SEQ_LEN}")

    # Avoid test-set leakage: use only the available initial prefix.
    # y[0] corresponds to raw[SEQ_LEN], so the first tinit targets
    # end at raw index SEQ_LEN + tinit - 1 (inclusive).
    scaler = StandardScaler()
    scaler.fit(raw[:SEQ_LEN + tinit].reshape(-1, 1))
    series = scaler.transform(raw.reshape(-1, 1)).ravel()
    X, y = build_sequences(series, SEQ_LEN)
    n_train = int(round(SPLIT * tinit))
    if n_train < 2 or n_train >= tinit:
        raise ValueError("Train/calibration split must leave both nonempty")

    device = DEVICE or ("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n{name}: raw={len(raw)} sequences={len(y)} tinit={tinit} "
          f"training={n_train} initial_cal={tinit-n_train} device={device}", flush=True)
    models, opts = setup_models(device)
    fit_models(models, opts, X[:n_train], y[:n_train], EPOCHS, device, ALPHA)

    # Initial calibration is temporal and held out from LSTM fitting.
    cal_lo, cal_hi = predict_quantiles(models, X[n_train:tinit], device)
    cal_scores = nonconformity(y[n_train:tinit], cal_lo, cal_hi)
    max_score = float(np.max(cal_scores))
    # Authors' scale rule applied to the modified quantile-interval scores.
    # Zero-only calibration needs a strictly positive numerical fallback.
    scale = max(max_score * math.sqrt(3), 1e-8)
    saocp = SAOCP(alpha=ALPHA, scale=scale, lifetime=LIFETIME)
    for score in cal_scores:
        saocp.update(float(score))

    rows = []
    for t in range(tinit, len(y)):
        # Keep the existing periodic base-model fine-tuning protocol.
        if t > tinit and (t - tinit) % TRAIN_EVERY == 0:
            a = t - tinit
            fit_models(models, opts, X[a:a+n_train], y[a:a+n_train],
                       1, device, ALPHA)

        lo_q, hi_q = predict_quantiles(models, X[t:t+1], device)
        lo_q, hi_q = float(lo_q[0]), float(hi_q[0])
        radius = saocp.predict()  # BEFORE seeing y[t]
        lower = lo_q - radius
        upper = hi_q + radius
        yt = float(y[t])
        err = int(yt < lower or yt > upper)
        score = float(max(lo_q - yt, yt - hi_q, 0.0))
        rows.append((t - tinit, t, yt, lower, upper, lo_q, hi_q,
                     radius, score, err, len(saocp.experts)))
        saocp.update(score)       # AFTER observing y[t]
        if LOG_EVERY and (t - tinit + 1) % LOG_EVERY == 0:
            print(f"  {name}: {t-tinit+1}/{len(y)-tinit} | "
                  f"radius={radius:.5f} | running_coverage="
                  f"{1.0-np.mean([r[9] for r in rows]):.4f}", flush=True)

    columns = ["step", "dataset_index", "y_true", "y_l_pred", "y_u_pred",
               "q_lower", "q_upper", "radius", "score", "err", "active_experts"]
    traj = pd.DataFrame(rows, columns=columns)
    for col in ["y_true", "y_l_pred", "y_u_pred", "q_lower", "q_upper", "radius", "score"]:
        # Transform score and radius as distances: multiply by std only.
        if col in {"radius", "score"}:
            traj[col] = traj[col] * float(scaler.scale_[0])
        else:
            traj[col] = scaler.inverse_transform(traj[[col]].to_numpy()).reshape(-1)
    traj["covered"] = 1 - traj["err"]
    traj["alphaTrajectory"] = np.nan  # SAOCP learns a radius, not alpha_t.
    traj["errSeq"] = traj["err"]
    rolling_local_trajectories(traj, LOCAL_WINDOW, ALPHA)

    outdir = Path(TRAJECTORY_DIR)
    outdir.mkdir(parents=True, exist_ok=True)
    if SAVE_TRAJECTORIES:
        traj.to_csv(outdir / f"{name}__SAOCP.csv", index=False)

    truth = traj["y_true"].to_numpy(dtype=float)
    lower = traj["y_l_pred"].to_numpy(dtype=float)
    upper = traj["y_u_pred"].to_numpy(dtype=float)
    elapsed = time.monotonic() - start_time
    # Identical evaluator, arguments and metric names as the ACI/LACI notebook.
    metrics = summarize_all_metrics(
        truth, lower, upper,
        alpha=ALPHA, w=LOCAL_WINDOW, eta=CWC_ETA, time_taken=elapsed
    )
    metrics["time_sec"] = float(elapsed)
    metrics["dataset"] = name + ".csv"
    metrics["method"] = "SAOCP"
    metrics["scale_D_from_cal"] = float(max_score * scaler.scale_[0])
    metrics["g_lifetime"] = LIFETIME
    metric_names = ["PICP", "MPIW", "MLPICP", "LPICE", "LPIA",
                    "LMPIW", "LCWC", "STD_LE"]
    print(f"[{name}/SAOCP] " + " | ".join(
        f"{k}={float(metrics[k]):.4f}" for k in metric_names if k in metrics
    ) + f" | time_sec={elapsed:.2f}", flush=True)
    return metrics


def self_test() -> None:
    # Pure NumPy/standard-library tests; no real datasets or GPU needed.
    np.random.seed(0)
    a = SAOCP(alpha=0.1, scale=math.sqrt(3), lifetime=8)
    scores = np.r_[np.full(70, 0.5), np.full(70, 2.0), np.full(70, 0.3)]
    rads = []
    for s in scores:
        r = a.predict()
        assert np.isfinite(r) and r >= 0
        rads.append(r)
        a.update(float(s))
        p = a.probabilities()
        assert abs(sum(p.values()) - 1.0) < 1e-9
        assert all(v >= 0.0 for v in p.values())
    assert len(a.experts) < 100
    assert np.mean(rads[100:140]) > np.mean(rads[25:60]), "Radius failed to respond to shift"
    assert np.allclose(nonconformity(np.array([2., 5., 7.]),
                                      np.array([1., 4., 4.]),
                                      np.array([3., 6., 6.])), [0., 0., 1.])
    # The nested set coverage event is score <= radius, including equality.
    for s in (0.0, 0.5, 2.0):
        for rad in (0.0, 0.5, 2.0):
            assert ((10.0 - rad <= 10.0 + s <= 10.0 + rad) == (s <= rad))
    print("Self-test passed: expert probabilities, finite radii, shift response, scores, interval inclusion")


def main() -> None:
    if RUN_SELF_TEST:
        self_test()
        return
    if TRAIN_EVERY < 1:
        raise ValueError("TRAIN_EVERY must be >= 1")
    if SCALER_MODE != "prefix":
        raise ValueError("Only leakage-free SCALER_MODE='prefix' is supported")
    if not DEFAULT_DATASETS:
        raise ValueError("DEFAULT_DATASETS must include at least one file")
    missing = [p for p in DEFAULT_DATASETS if not Path(p).is_file()]
    if missing:
        raise FileNotFoundError("Datasets missing (no synthetic fallback):\n" + "\n".join(missing))
    metrics = []
    for path in DEFAULT_DATASETS:
        row = run_dataset(path)
        metrics.append(row)
        pd.DataFrame(metrics).to_csv(OUTPUT_CSV, index=False)
    # Match the column ordering in your ACI/DtACI/AgACI/LACI notebook.
    results_df = pd.DataFrame(metrics)
    metric_order = ["dataset", "method", "PICP", "MPIW", "MLPICP", "LPICE",
                    "LPIA", "LMPIW", "LCWC", "STD_LE", "n_local_windows",
                    "time_sec"]
    ordered = [k for k in metric_order if k in results_df.columns]
    results_df = results_df[ordered + [k for k in results_df if k not in ordered]]
    results_df.to_csv(OUTPUT_CSV, index=False)
    print("\nSaved summary:", OUTPUT_CSV)
    if SAVE_TRAJECTORIES:
        print("Saved trajectories:", TRAJECTORY_DIR)
    print(results_df.to_string(index=False))


if __name__ == "__main__":
    main()



Plant_1_Generation_Data: raw=7379 sequences=7331 tinit=4000 training=2000 initial_cal=2000 device=cuda
  Plant_1_Generation_Data: 200/3331 | radius=0.52582 | running_coverage=0.9000
  Plant_1_Generation_Data: 400/3331 | radius=0.28447 | running_coverage=0.9050
  Plant_1_Generation_Data: 600/3331 | radius=0.00000 | running_coverage=0.9217
  Plant_1_Generation_Data: 800/3331 | radius=0.00000 | running_coverage=0.9363
  Plant_1_Generation_Data: 1000/3331 | radius=0.00000 | running_coverage=0.9460
  Plant_1_Generation_Data: 1200/3331 | radius=0.05534 | running_coverage=0.9442
  Plant_1_Generation_Data: 1400/3331 | radius=0.01913 | running_coverage=0.9414
  Plant_1_Generation_Data: 1600/3331 | radius=0.04376 | running_coverage=0.9400
  Plant_1_Generation_Data: 1800/3331 | radius=0.02100 | running_coverage=0.9389
  Plant_1_Generation_Data: 2000/3331 | radius=0.22112 | running_coverage=0.9350
  Plant_1_Generation_Data: 2200/3331 | radius=0.00000 | running_coverage=0.9377
  Plant_1_Generation

In [4]:
"""SAOCP-only evaluation on the hydropower dataset, matching the supplied ACI setup.

Implementation of Bhatnagar, Wang, Xiong & Bai (ICML 2023), Algorithm 1
(SAOCP) with Algorithm 2 (SF-OGD) and the implementation choices used by
https://github.com/salesforce/online_conformal/blob/main/online_conformal/saocp.py

Changes from the paper's forecasting experiment:
  - Base model: user's rolling LSTM lower/upper quantile regressors.
  - Nested prediction set: [q_lower - radius, q_upper + radius].
  - True nonnegative score: max(q_lower-y, y-q_upper, 0).
  - One-step-ahead sequential evaluation rather than M4 batched forecasts.
  - Scale selected from initial calibration residuals as in the authors' code.

Run from your original experiment directory, with Datasets and Helper_Scripts
in their original relative locations:
    python run_saocp_hydropower_corrected.py
    # Edit DEFAULT_DATASETS and configuration constants below to customize.

The actual datasets and Helper_Scripts are not embedded in the uploaded .ipynb.
Requires numpy, pandas, scikit-learn, torch, and your models_TLMR.py.
The original Helper_Scripts.calculate_metrics.py is REQUIRED for metric consistency.
This is SAOCP adapted to quantile-interval expansion, not an exact replication
of the original point-forecast experiments. The filtering of missing/zero
hydropower observations matches the supplied experiment and may compress time.
"""
from __future__ import annotations

import math
import random
import time
import sys
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd


SEED = 42
SEQ_LEN = 48
EPOCHS = 200
LR = 0.001
BATCH_SIZE = 32
ALPHA = 0.10
SPLIT = 0.50
LIFETIME = 8              # g in paper Eq. (8), authors' official default
TRAIN_EVERY = 6           # preserve notebook update cadence
SAVE_TRAJECTORIES = True
# All settings are configured here. No command-line arguments are required.
TINIT_OVERRIDE = 200        # same fixed TINIT as the hydropower experiment
SCALER_MODE = "prefix"     # fit only on historical observations; no future leakage
DEVICE = None             # None selects CUDA automatically when available
LOG_EVERY = 200

# The local window and CWC eta are imported from the SAME helper as ACI/LACI.
# Do not define alternate local-metric formulas here.
TRAJECTORY_DIR = "./trajectories/LSTM_SAOCP_hydropower"
OUTPUT_CSV = "./results_SAOCP_hydropower.csv"
RUN_SELF_TEST = False      # True runs internal algorithm checks instead of experiments

DEFAULT_DATASETS = [
    "../Datasets/hydropower_dataset.csv",
]

HYDRO_NAMES = {"hydropower_dataset"}


def pinball(y: float, s: float, coverage: float) -> float:
    """Paper (1): (1-alpha)-pinball loss with y=true score, s=radius."""
    return max(coverage * (y - s), (1.0 - coverage) * (s - y))


def pinball_grad(y: float, s: float, coverage: float) -> float:
    """Authors' tie convention: zero subgradient when y == s."""
    return (-coverage if y > s else (1.0 - coverage if y < s else 0.0))


def two_adic_valuation(n: int) -> int:
    assert n >= 1
    return (n & -n).bit_length() - 1


@dataclass
class SFOGDExpert:
    """Individual SAOCP expert, based on the authors' _OGD implementation."""
    start: int
    scale: float
    alpha: float
    radius: float
    g: int
    grad_norm: float = 0.0
    z: float = 0.0
    wz: float = 0.0
    age: int = 0

    @property
    def lifetime(self) -> int:
        return self.g * (2 ** two_adic_valuation(self.start))

    @property
    def expired(self) -> bool:
        # Matches author's code: expired when age > lifetime.
        return self.age > self.lifetime

    @property
    def wealth_weight(self) -> float:
        return 0.0 if self.age == 0 else (self.z / self.age) * (1.0 + self.wz)

    def update(self, score: float, meta_loss: float) -> None:
        old_weight = self.wealth_weight
        loss_expert = pinball(score, self.radius, 1.0 - self.alpha)
        # Matches official _OGD.update, including its normalization and clipping.
        gain = (meta_loss - loss_expert) / self.scale / max(self.alpha, 1.0 - self.alpha)
        gain = float(np.clip(gain, -float(old_weight > 0.0), 1.0))
        self.z += gain
        self.wz += gain * old_weight
        self.age += 1
        grad = pinball_grad(score, self.radius, 1.0 - self.alpha)
        self.grad_norm += grad * grad
        if self.grad_norm != 0.0:
            eta = self.scale / math.sqrt(3.0)
            self.radius = max(0.0, self.radius - eta * grad / math.sqrt(self.grad_norm))


class SAOCP:
    """One-step SAOCP, faithful to salesforce/online_conformal/saocp.py.

    Call predict() before seeing the current y. Then call update(score).
    update() creates an expert at the current time, aggregates meta loss,
    updates active experts and increments time; it never sees future scores.
    """
    def __init__(self, alpha: float, scale: float, lifetime: int = LIFETIME):
        if not 0.0 < alpha < 1.0:
            raise ValueError("alpha must lie strictly between zero and one")
        if not np.isfinite(scale) or scale <= 0.0:
            raise ValueError("scale must be finite and positive")
        if lifetime < 1:
            raise ValueError("lifetime must be positive")
        self.alpha = float(alpha)
        self.scale = float(scale)
        self.lifetime = int(lifetime)
        self.t = 1
        self.experts: dict[int, SFOGDExpert] = {}

    def probabilities(self) -> dict[int, float]:
        if not self.experts:
            return {}
        prior = {
            k: 1.0 / (k * k * (1.0 + math.floor(math.log2(k))))
            for k in self.experts
        }
        total_prior = sum(prior.values())
        prior = {k: v / total_prior for k, v in prior.items()}
        unnorm = {k: prior[k] * max(0.0, e.wealth_weight)
                  for k, e in self.experts.items()}
        total = sum(unnorm.values())
        return ({k: v / total for k, v in unnorm.items()}
                if total > 0.0 else prior)

    def predict(self) -> float:
        p = self.probabilities()
        return float(sum(p[k] * self.experts[k].radius for k in p))

    def update(self, score: float) -> None:
        if not np.isfinite(score) or score < 0.0:
            raise ValueError(f"Expected a nonnegative finite conformity score: {score}")
        # Official code initializes the new expert from the CURRENT aggregate,
        # then aggregates once more INCLUDING the newly created expert.
        init_radius = self.predict()
        for k in [k for k, v in self.experts.items() if v.expired]:
            del self.experts[k]
        self.experts[self.t] = SFOGDExpert(
            start=self.t, scale=self.scale, alpha=self.alpha,
            radius=init_radius, g=self.lifetime)
        meta_radius = self.predict()
        meta_loss = pinball(float(score), meta_radius, 1.0 - self.alpha)
        for expert in self.experts.values():
            expert.update(float(score), meta_loss)
        self.t += 1


def nonconformity(y: np.ndarray, lower: np.ndarray, upper: np.ndarray) -> np.ndarray:
    """Smallest nonnegative expansion of [lower, upper] covering y."""
    return np.maximum.reduce([lower - y, y - upper, np.zeros_like(y)])


def read_series(filepath: str) -> np.ndarray:
    """Match hydro ACI: sum both actual peak columns, remove nulls and zeros."""
    path = Path(filepath)
    if not path.is_file():
        raise FileNotFoundError(f"Missing dataset: {path}")
    if path.stem not in HYDRO_NAMES:
        raise ValueError(f"Only the hydropower dataset is supported: {path}")
    df = pd.read_csv(path)
    required = ["Actual Day Peak (MW)", "Actual Eve Peak (MW)"]
    missing = [column for column in required if column not in df.columns]
    if missing:
        raise ValueError(f"{path}: missing required columns: {missing}")
    # pandas addition matches the supplied data loader: NaN propagates.
    day = pd.to_numeric(df[required[0]], errors="coerce")
    evening = pd.to_numeric(df[required[1]], errors="coerce")
    df["total_power_output"] = day + evening
    value = df["total_power_output"]
    y = value.loc[value.notna() & (value != 0)].to_numpy(dtype=float)
    if not len(y) or not np.all(np.isfinite(y)):
        raise ValueError(f"Nonfinite or empty filtered total_power_output series: {path}")
    return y


def build_sequences(series: np.ndarray, seq_len: int) -> tuple[np.ndarray, np.ndarray]:
    # A sliding view avoids the very slow Python list construction in notebook.
    if len(series) <= seq_len:
        raise ValueError("Series too short for SEQ_LEN")
    windows = np.lib.stride_tricks.sliding_window_view(series, seq_len + 1)
    X = np.ascontiguousarray(windows[:, :-1], dtype=np.float32)
    y = np.asarray(windows[:, -1], dtype=np.float32)
    return X, y


def seed_everything(seed: int) -> None:
    import torch
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def setup_models(device: str):
    import torch
    if str(Path.cwd()) not in sys.path:
        sys.path.insert(0, str(Path.cwd()))
    try:
        from Helper_Scripts.models_TLMR import LSTMQuantileReg
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            "Missing Helper_Scripts.models_TLMR.LSTMQuantileReg; run alongside "
            "your existing experiment or adjust import in setup_models()."
        ) from exc
    models = [LSTMQuantileReg().to(device) for _ in range(2)]
    opts = [torch.optim.Adam(m.parameters(), lr=LR) for m in models]
    return models, opts


def fit_models(models, opts, X: np.ndarray, y: np.ndarray, epochs: int,
               device: str, alpha: float) -> None:
    import torch
    from torch.utils.data import DataLoader, TensorDataset
    if epochs < 1:
        return
    data = TensorDataset(torch.as_tensor(X, dtype=torch.float32),
                         torch.as_tensor(y.reshape(-1, 1), dtype=torch.float32))
    loader = DataLoader(data, batch_size=min(BATCH_SIZE, len(data)), shuffle=True)
    for model, opt, q in zip(models, opts, [alpha / 2.0, 1.0 - alpha / 2.0]):
        model.train()
        for _ in range(epochs):
            for bx, by in loader:
                bx, by = bx.to(device), by.to(device)
                pred = model(bx)
                diff = by - pred
                loss = torch.maximum(q * diff, (q - 1.0) * diff).mean()
                opt.zero_grad()
                loss.backward()
                opt.step()


def predict_quantiles(models, X: np.ndarray, device: str,
                      batch_size: int = 4096) -> tuple[np.ndarray, np.ndarray]:
    import torch
    outputs = []
    for model in models:
        model.eval()
        chunks = []
        with torch.no_grad():
            for idx in range(0, len(X), batch_size):
                bx = torch.as_tensor(X[idx:idx + batch_size], dtype=torch.float32,
                                     device=device)
                chunks.append(model(bx).detach().cpu().numpy().reshape(-1))
        outputs.append(np.concatenate(chunks).astype(float))
    # Crossing correction to guarantee a valid nested interval family.
    return np.minimum(outputs[0], outputs[1]), np.maximum(outputs[0], outputs[1])


def rolling_local_trajectories(traj: pd.DataFrame, window: int, alpha: float) -> None:
    """Store per-time local diagnostics without changing the metric definitions.

    Only complete, trailing windows are reported (the first window-1 rows
    remain NaN), matching the usual local-coverage evaluation convention.
    Summary metrics are computed exclusively by the original shared helper.
    """
    if window < 1:
        raise ValueError("WINDOW from calculate_metrics must be positive")
    covered = ((traj["y_l_pred"] <= traj["y_true"]) &
               (traj["y_true"] <= traj["y_u_pred"])).astype(float)
    width = traj["y_u_pred"] - traj["y_l_pred"]
    lp = covered.rolling(window=window, min_periods=window).mean()
    traj["LPICP"] = lp
    traj["LE"] = ((1.0 - alpha) - lp).clip(lower=0.0)
    traj["local_overcoverage"] = (lp - (1.0 - alpha)).clip(lower=0.0)
    traj["local_abs_coverage_error"] = (lp - (1.0 - alpha)).abs()
    traj["local_MPIW"] = width.rolling(window=window, min_periods=window).mean()


def run_dataset(filepath: str) -> dict:
    from sklearn.preprocessing import StandardScaler
    from Helper_Scripts.calculate_metrics import (
        summarize_all_metrics, WINDOW as LOCAL_WINDOW, CWC_ETA
    )
    import torch
    name = Path(filepath).stem
    start_time = time.monotonic()
    seed_everything(SEED)
    raw = read_series(filepath)
    tinit = TINIT_OVERRIDE
    if len(raw) <= SEQ_LEN + tinit:
        raise ValueError(f"{name}: {len(raw)} points cannot support tinit={tinit}, seq_len={SEQ_LEN}")

    # X[:tinit], y[:tinit] use raw indices 0 through SEQ_LEN+tinit-1.
    # Fit the scaler exclusively on these observations, never future test data.
    scaler = StandardScaler()
    scaler.fit(raw[:SEQ_LEN + tinit].reshape(-1, 1))
    series = scaler.transform(raw.reshape(-1, 1)).ravel()
    X, y = build_sequences(series, SEQ_LEN)
    n_train = int(round(SPLIT * tinit))
    if n_train < 2 or n_train >= tinit:
        raise ValueError("Train/calibration split must leave both nonempty")

    device = DEVICE or ("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n{name}: raw={len(raw)} sequences={len(y)} tinit={tinit} "
          f"training={n_train} initial_cal={tinit-n_train} device={device}", flush=True)
    models, opts = setup_models(device)
    fit_models(models, opts, X[:n_train], y[:n_train], EPOCHS, device, ALPHA)

    # Initial calibration is temporal and held out from LSTM fitting.
    cal_lo, cal_hi = predict_quantiles(models, X[n_train:tinit], device)
    cal_scores = nonconformity(y[n_train:tinit], cal_lo, cal_hi)
    max_score = float(np.max(cal_scores))
    # Authors' implementation: scale = max|residual| * sqrt(3).
    # Zero-only calibration needs a strictly positive numerical fallback.
    scale = max(max_score * math.sqrt(3), 1e-8)
    saocp = SAOCP(alpha=ALPHA, scale=scale, lifetime=LIFETIME)
    for score in cal_scores:
        saocp.update(float(score))

    rows = []
    for t in range(tinit, len(y)):
        # Rolling window exactly as notebook: train is first fraction of past
        # tinit examples; the remaining fraction is not used for online radius
        # learning because SAOCP directly consumes sequential test scores.
        if t > tinit and (t - tinit) % TRAIN_EVERY == 0:
            # Retain the original rolling-window convention. All targets in
            # this training slice precede t (the current forecast target).
            a = t - tinit
            assert a + n_train <= t
            fit_models(models, opts, X[a:a+n_train], y[a:a+n_train],
                       1, device, ALPHA)

        lo_q, hi_q = predict_quantiles(models, X[t:t+1], device)
        lo_q, hi_q = float(lo_q[0]), float(hi_q[0])
        radius = saocp.predict()  # BEFORE seeing y[t]
        lower = lo_q - radius
        upper = hi_q + radius
        yt = float(y[t])
        err = int(yt < lower or yt > upper)
        score = float(max(lo_q - yt, yt - hi_q, 0.0))
        rows.append((t - tinit, t, yt, lower, upper, lo_q, hi_q,
                     radius, score, err, len(saocp.experts)))
        saocp.update(score)       # AFTER observing y[t]
        if LOG_EVERY and (t - tinit + 1) % LOG_EVERY == 0:
            print(f"  {name}: {t-tinit+1}/{len(y)-tinit} | "
                  f"radius={radius:.5f} | running_coverage="
                  f"{1.0-np.mean([r[9] for r in rows]):.4f}", flush=True)

    columns = ["step", "dataset_index", "y_true", "y_l_pred", "y_u_pred",
               "q_lower", "q_upper", "radius", "score", "err", "active_experts"]
    traj = pd.DataFrame(rows, columns=columns)
    for col in ["y_true", "y_l_pred", "y_u_pred", "q_lower", "q_upper", "radius", "score"]:
        # Transform score and radius as distances: multiply by std only.
        if col in {"radius", "score"}:
            traj[col] = traj[col] * float(scaler.scale_[0])
        else:
            traj[col] = scaler.inverse_transform(traj[[col]].to_numpy()).reshape(-1)
    traj["covered"] = 1 - traj["err"]
    traj["alphaTrajectory"] = np.nan  # SAOCP learns a radius, not alpha_t.
    traj["errSeq"] = traj["err"]
    rolling_local_trajectories(traj, LOCAL_WINDOW, ALPHA)

    outdir = Path(TRAJECTORY_DIR)
    outdir.mkdir(parents=True, exist_ok=True)
    if SAVE_TRAJECTORIES:
        traj.to_csv(outdir / f"{name}__SAOCP.csv", index=False)

    truth = traj["y_true"].to_numpy(dtype=float)
    lower = traj["y_l_pred"].to_numpy(dtype=float)
    upper = traj["y_u_pred"].to_numpy(dtype=float)
    elapsed = time.monotonic() - start_time
    # Identical evaluator, arguments and metric names as the ACI/LACI notebook.
    metrics = summarize_all_metrics(
        truth, lower, upper,
        alpha=ALPHA, w=LOCAL_WINDOW, eta=CWC_ETA, time_taken=elapsed
    )
    metrics["time_sec"] = float(elapsed)
    metrics["dataset"] = name + ".csv"
    metrics["method"] = "SAOCP"
    metrics["scale_D_from_cal"] = float(max_score * scaler.scale_[0])
    metrics["g_lifetime"] = LIFETIME
    metric_names = ["PICP", "MPIW", "MLPICP", "LPICE", "LPIA",
                    "LMPIW", "LCWC", "STD_LE"]
    print(f"[{name}/SAOCP] " + " | ".join(
        f"{k}={float(metrics[k]):.4f}" for k in metric_names if k in metrics
    ) + f" | time_sec={elapsed:.2f}", flush=True)
    return metrics


def self_test() -> None:
    # Pure NumPy/standard-library tests; no real datasets or GPU needed.
    np.random.seed(0)
    a = SAOCP(alpha=0.1, scale=math.sqrt(3), lifetime=8)
    scores = np.r_[np.full(70, 0.5), np.full(70, 2.0), np.full(70, 0.3)]
    rads = []
    for s in scores:
        r = a.predict()
        assert np.isfinite(r) and r >= 0
        rads.append(r)
        a.update(float(s))
        p = a.probabilities()
        assert abs(sum(p.values()) - 1.0) < 1e-9
        assert all(v >= 0.0 for v in p.values())
    assert len(a.experts) < 100
    assert np.mean(rads[100:140]) > np.mean(rads[25:60]), "Radius failed to respond to shift"
    assert np.allclose(nonconformity(np.array([2., 5., 7.]),
                                      np.array([1., 4., 4.]),
                                      np.array([3., 6., 6.])), [0., 0., 1.])
    # Test causal timing and non-negative interval-expansion score.
    assert nonconformity(np.array([10.0]), np.array([8.0]),
                         np.array([12.0]))[0] == 0.0
    assert all(np.isfinite(r) and r >= 0 for r in rads)
    # The nested set coverage event is score <= radius, including equality.
    for s in (0.0, 0.5, 2.0):
        for rad in (0.0, 0.5, 2.0):
            assert ((10.0 - rad <= 10.0 + s <= 10.0 + rad) == (s <= rad))
    print("Self-test passed: expert probabilities, finite radii, shift response, scores, interval inclusion")


def main() -> None:
    if RUN_SELF_TEST:
        self_test()
        return
    if TRAIN_EVERY < 1:
        raise ValueError("TRAIN_EVERY must be >= 1")
    if SCALER_MODE != "prefix":
        raise ValueError("Only leakage-free SCALER_MODE='prefix' is supported")
    if not DEFAULT_DATASETS:
        raise ValueError("DEFAULT_DATASETS must include at least one file")
    missing = [p for p in DEFAULT_DATASETS if not Path(p).is_file()]
    if missing:
        raise FileNotFoundError("Datasets missing (no synthetic fallback):\n" + "\n".join(missing))
    metrics = []
    for path in DEFAULT_DATASETS:
        row = run_dataset(path)
        metrics.append(row)
        pd.DataFrame(metrics).to_csv(OUTPUT_CSV, index=False)
    # Match the column ordering in your ACI/DtACI/AgACI/LACI notebook.
    results_df = pd.DataFrame(metrics)
    metric_order = ["dataset", "method", "PICP", "MPIW", "MLPICP", "LPICE",
                    "LPIA", "LMPIW", "LCWC", "STD_LE", "n_local_windows",
                    "time_sec"]
    ordered = [k for k in metric_order if k in results_df.columns]
    results_df = results_df[ordered + [k for k in results_df if k not in ordered]]
    results_df.to_csv(OUTPUT_CSV, index=False)
    print("\nSaved summary:", OUTPUT_CSV)
    if SAVE_TRAJECTORIES:
        print("Saved trajectories:", TRAJECTORY_DIR)
    print(results_df.to_string(index=False))


if __name__ == "__main__":
    main()



hydropower_dataset: raw=1753 sequences=1705 tinit=200 training=100 initial_cal=100 device=cuda
  hydropower_dataset: 200/1505 | radius=0.71214 | running_coverage=0.9200
  hydropower_dataset: 400/1505 | radius=0.00000 | running_coverage=0.9425
  hydropower_dataset: 600/1505 | radius=0.12633 | running_coverage=0.9400
  hydropower_dataset: 800/1505 | radius=0.27584 | running_coverage=0.9350
  hydropower_dataset: 1000/1505 | radius=0.11988 | running_coverage=0.9410
  hydropower_dataset: 1200/1505 | radius=0.93005 | running_coverage=0.9392
  hydropower_dataset: 1400/1505 | radius=0.07699 | running_coverage=0.9443
[hydropower_dataset/SAOCP] PICP=0.9429 | MPIW=181.6568 | MLPICP=0.9437 | LPICE=0.0061 | LPIA=0.8219 | LMPIW=180.3503 | LCWC=5.6840 | STD_LE=0.0156 | time_sec=5.97

Saved summary: ./results_SAOCP_hydropower.csv
Saved trajectories: ./trajectories/LSTM_SAOCP_hydropower
               dataset method     PICP       MPIW   MLPICP    LPICE     LPIA      LMPIW     LCWC   STD_LE  n_local_w